# PaddleOCR Test — CNIE Field Crops

Uses the already-cropped `front_cropped.jpg` (856x540).
Tests PaddleOCR with `det=False` (recognition only) on each field region.

**No preprocessing needed** — raw BGR crops go straight into the transformer model.

In [ ]:
# Run ONCE to install — PaddlePaddle FIRST, then PaddleOCR
!python -m pip install paddlepaddle==3.0.0 -i https://www.paddlepaddle.org.cn/packages/stable/cpu/
!python -m pip install paddleocr
!pip install shapely scikit-image "protobuf>=3.20.0,<4.0"

In [ ]:
# Verify installation
import paddle
print(f"PaddlePaddle version: {paddle.__version__}")

from paddleocr import PaddleOCR
print("PaddleOCR imported OK")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time

# Load the already-cropped card
front = cv2.imread('front_cropped.jpg')
print(f'Front loaded: {front.shape[1]}w x {front.shape[0]}h')

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(front, cv2.COLOR_BGR2RGB))
plt.title(f'Cropped Card — {front.shape[1]}x{front.shape[0]}')
plt.axis('off')
plt.show()

---
## Initialize OCR Instances

Two instances: one for Arabic, one for French/English.
Models download on first run (~10-40MB each, cached in `~/.paddleocr/`).

In [ ]:
# Initialize ONCE — heavy step, ~5-10s per instance (model download + load)
print("Loading Arabic model...")
t0 = time.time()
ocr_ar = PaddleOCR(
    use_angle_cls=False,
    lang='ar',
    use_gpu=False,
    ocr_version='PP-OCRv4',
    drop_score=0.3,
    show_log=False
)
print(f"Arabic model ready ({time.time()-t0:.1f}s)")

print("Loading French model...")
t0 = time.time()
ocr_fr = PaddleOCR(
    use_angle_cls=False,
    lang='fr',
    use_gpu=False,
    ocr_version='PP-OCRv4',
    drop_score=0.3,
    show_log=False
)
print(f"French model ready ({time.time()-t0:.1f}s)")

print("Loading English model (for dates/numbers)...")
t0 = time.time()
ocr_en = PaddleOCR(
    use_angle_cls=False,
    lang='en',
    use_gpu=False,
    ocr_version='PP-OCRv4',
    drop_score=0.3,
    show_log=False
)
print(f"English model ready ({time.time()-t0:.1f}s)")

print("\nAll models loaded. Ready to OCR.")

---
## Field Definitions + OCR Helper

In [ ]:
PADDING = 10

FRONT_FIELDS = {
    "first_name_fr":      {"x": 0,   "y": 148, "w": 500, "h": 52,  "ocr": "fr"},
    "last_name_fr":       {"x": 0,   "y": 218, "w": 500, "h": 50,  "ocr": "fr"},
    "date_of_birth":      {"x": 160, "y": 255, "w": 210, "h": 54,  "ocr": "en"},
    "place_of_birth_fr":  {"x": 0,   "y": 322, "w": 500, "h": 50,  "ocr": "fr"},
    "expiry_date":        {"x": 200, "y": 365, "w": 200, "h": 45,  "ocr": "en"},
    "first_name_ar":      {"x": 330, "y": 130, "w": 270, "h": 48,  "ocr": "ar"},
    "last_name_ar":       {"x": 330, "y": 204, "w": 270, "h": 48,  "ocr": "ar"},
    "card_number":        {"x": 590, "y": 405, "w": 220, "h": 42,  "ocr": "en"},
    "gender":             {"x": 800, "y": 400, "w": 56,  "h": 45,  "ocr": "en"},
}

# Map language to OCR instance
OCR_INSTANCES = {
    "ar": ocr_ar,
    "fr": ocr_fr,
    "en": ocr_en,
}


def paddle_ocr_field(img, x, y, w, h, field_name, ocr_lang, padding=PADDING):
    """
    Crop a field region and run PaddleOCR recognition (no detection).
    Raw BGR crop goes straight in — no preprocessing needed.
    """
    ih, iw = img.shape[:2]
    x1 = max(0, x - padding)
    y1 = max(0, y - padding)
    x2 = min(iw, x + w + padding)
    y2 = min(ih, y + h + padding)

    crop = img[y1:y2, x1:x2]

    if crop.size == 0:
        print(f"ERROR: Empty crop for {field_name}")
        return None

    # Get the right OCR instance
    ocr = OCR_INSTANCES[ocr_lang]

    # Recognition only — det=False since we already cropped
    t0 = time.time()
    result = ocr.ocr(crop, det=False, rec=True, cls=False)
    elapsed = time.time() - t0

    # Parse result
    text = ""
    conf = 0.0
    if result and result[0]:
        texts = [item[0] for item in result[0]]
        scores = [item[1] for item in result[0]]
        text = " ".join(texts)
        conf = min(scores) if scores else 0.0

    # Display
    plt.figure(figsize=(10, 1.5))
    plt.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    plt.title(f"{field_name}  |  lang={ocr_lang}  |  '{text}'  |  conf={conf:.2f}  |  {elapsed*1000:.0f}ms",
              fontsize=10, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    return {"text": text, "conf": conf, "time_ms": elapsed * 1000}


print(f"Defined {len(FRONT_FIELDS)} fields. Ready to test.")

---
## Test All Front Side Fields

In [ ]:
print(f"Testing {len(FRONT_FIELDS)} fields with PaddleOCR (det=False, raw BGR crops)")
print(f"Padding: {PADDING}px")
print("=" * 60)

all_results = {}
total_time = 0

for field_name, field in FRONT_FIELDS.items():
    result = paddle_ocr_field(
        front,
        field["x"], field["y"], field["w"], field["h"],
        field_name,
        ocr_lang=field["ocr"],
        padding=PADDING
    )
    if result:
        all_results[field_name] = result
        total_time += result["time_ms"]

# Summary
print("\n" + "=" * 60)
print("SUMMARY — PaddleOCR PP-OCRv4")
print("=" * 60)
for name, r in all_results.items():
    status = ">>" if r["conf"] > 0.5 else "!!"
    print(f"  {status} {name:20s} -> '{r['text']}'  (conf={r['conf']:.2f}, {r['time_ms']:.0f}ms)")

print(f"\nTotal OCR time: {total_time:.0f}ms for {len(all_results)} fields")
print(f"Average: {total_time/len(all_results):.0f}ms per field")

---
## Bonus: Try PP-OCRv5 (If v4 Results Are Weak)

PP-OCRv5 has ~40% accuracy improvement on some scripts. Just change `ocr_version`.

In [ ]:
# Uncomment and run this cell to test PP-OCRv5
# Only re-initialize the models that had weak results above

# print("Loading PP-OCRv5 Arabic model...")
# ocr_ar_v5 = PaddleOCR(
#     use_angle_cls=False,
#     lang='ar',
#     use_gpu=False,
#     ocr_version='PP-OCRv5',
#     drop_score=0.3,
#     show_log=False
# )
# print("Ready. Now re-run the test cell above after changing:")
# print("  OCR_INSTANCES['ar'] = ocr_ar_v5")

---
## Compare: PaddleOCR vs Tesseract

Fill in after running:

| Field | Tesseract | PaddleOCR | Winner |
|-------|-----------|-----------|--------|
| first_name_fr | ? | ? | ? |
| last_name_fr | ? | ? | ? |
| date_of_birth | ? | ? | ? |
| place_of_birth_fr | ? | ? | ? |
| expiry_date | ? | ? | ? |
| first_name_ar | ? | ? | ? |
| last_name_ar | ? | ? | ? |
| card_number | ? | ? | ? |
| gender | ? | ? | ? |

**Speed:** Tesseract ~300ms/field vs PaddleOCR ~50ms/field

**Decision:** ___